# 📊 Data Preparation for Price Elasticity Models
---

## Objective

Prepare training data for demand estimation following best practices in applied econometrics.

## Pipeline

```
RAW (Snowflake)           PROCESSED (DataFrame)        TRAINING READY
─────────────────        ─────────────────────        ─────────────────
SALES_ORDER      ──┐
PRODUCT          ──┼──▶  Feature Engineering  ──▶   ln(Q), ln(P)
COST_TO_SERVE    ──┤                                 Lagged prices
MARKETPLACE_CPI  ──┘                                 Market indices
```

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print('✅ Environment ready')

---
## 1. Data Ingestion

In [ ]:
df = session.sql('SELECT * FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA').to_pandas()

print('📊 DATASET OVERVIEW')
print('=' * 50)
print(f'Observations:     {len(df):>10,}')
print(f'Date Range:       {df["ORDER_DATE"].min()} to {df["ORDER_DATE"].max()}')
print(f'Products:         {df["PRODUCT_ID"].nunique():>10}')
print(f'Product Families: {df["PRODUCT_FAMILY"].nunique():>10}')
print(f'Regions:          {df["REGION"].nunique():>10}')

---
## 2. Data Quality Assessment

### 2.1 Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality = pd.DataFrame({'Missing': missing, 'Missing %': missing_pct})
print('📋 MISSING VALUES')
print(quality[quality['Missing'] > 0].sort_values('Missing %', ascending=False))

### 2.2 Descriptive Statistics

| Variable | Description | Expected Sign |
|----------|-------------|---------------|
| `LN_QUANTITY` | Log of quantity sold | Dependent variable |
| `LN_PRICE` | Log of price | Negative (law of demand) |
| `FUEL_OIL_CPI` | Energy cost index | Positive (cost pass-through) |

In [ ]:
key_vars = ['LN_QUANTITY', 'LN_PRICE', 'MARGIN_PCT', 'FUEL_OIL_CPI', 'INDUSTRIAL_PRODUCTION']
print('\n📈 DESCRIPTIVE STATISTICS')
print(df[key_vars].describe().round(4).T[['count','mean','std','min','50%','max']])

---
## 3. Exploratory Data Analysis

### 3.1 Price-Quantity Relationship

The **law of demand** predicts a negative correlation: ↑P → ↓Q

In [ ]:
# Scatter with OLS trendlines
fig = px.scatter(
    df.dropna(subset=['LN_PRICE','LN_QUANTITY']).sample(min(5000,len(df))),
    x='LN_PRICE', y='LN_QUANTITY', color='PRODUCT_FAMILY',
    opacity=0.4, trendline='ols',
    title='<b>Demand Curves by Product Family</b><br><sup>Log-log specification: slope ≈ price elasticity</sup>',
    labels={'LN_PRICE': 'ln(Price per MT)', 'LN_QUANTITY': 'ln(Quantity MT)'}
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

r, p = stats.pearsonr(df['LN_PRICE'].dropna(), df['LN_QUANTITY'].dropna())
print(f'\n🔍 Correlation: ρ = {r:.4f} (p = {p:.2e})')

### 3.2 Time Series Patterns

In [ ]:
ts = df.groupby(['ORDER_DATE','PRODUCT_FAMILY']).agg({'AVG_PRICE_PER_MT':'mean'}).reset_index()
fig = px.line(ts, x='ORDER_DATE', y='AVG_PRICE_PER_MT', color='PRODUCT_FAMILY',
              title='<b>Price Trends by Product Family</b>')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

### 3.3 Distribution Analysis

In [ ]:
fig = px.box(df, x='PRODUCT_FAMILY', y='LN_PRICE', color='PRODUCT_FAMILY',
             title='<b>Log Price Distribution by Family</b>')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

---
## 4. Train/Test Split

**Strategy:** Time-based split (80/20) to respect temporal ordering

In [ ]:
dates = df['ORDER_DATE'].sort_values().unique()
split_date = dates[int(len(dates) * 0.8)]

train = df[df['ORDER_DATE'] < split_date]
test = df[df['ORDER_DATE'] >= split_date]

print('📊 TRAIN/TEST SPLIT')
print(f'Split Date: {split_date}')
print(f'Train: {len(train):,} observations ({len(train)/len(df)*100:.1f}%)')
print(f'Test:  {len(test):,} observations ({len(test)/len(df)*100:.1f}%)')

---
## ✅ Data Preparation Complete

**Next Steps:**
1. Proceed to `02_linear_elasticity` for baseline OLS estimation
2. Then `03_sur_elasticity` for cross-price effects
3. Finally `04_optimal_pricing` for portfolio optimization